# Evaluate Term Dispersion Scores on the BC5CDR Corpus Data and Reproduce Results Reported in the Manuscript 

Description: Evaluate the following term dispersion score/keyword extraction methods on the BC5CDR corpus data:
- Inverse Document Frequency (IDF)
- Inverse Collection Frequency (ICF)
- Chi-square
- Church and Gale (CG)
- Irvine and Callison-Burch (ICB)
- Derivation of Proportions (DoP)
- Residual ICF (RICF)
- KeyBERT
- KeyLLM

Calculate average P@k scores for each scoring function using the BC5CDR terms as ground truth. Also, evaluate scoring functions for their ability to filter out stopwords.

This version of the code excludes singletons in the analysis.

## Preliminaries

In [1]:
# Imports
import sys
import os
import pickle
import json
import pandas as pd
sys.path.append('../../../')
import wordstats
from sklearn.feature_extraction.text import CountVectorizer
import random
import numpy as np
import scipy
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
from io import StringIO
from numpy import nan
from tqdm import tqdm
import rbo

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/pasheridan/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Load the BC5CDR Corpus Data

In particular, we load the preprocessed BC5CDR corpus documents, and gold standard biological terms (i.e., lexical units) and their associated semantic classes (i.e., sems) and associated high-level class (i.e., Chemical and Disease).

First, load the corpus docs, and the lexical units. Then hardcode the high-level semantic classes.

In [2]:
# Load the preprocessed BC5CDR corpus documents
bc5cdr_corpus_path = '../../1-preprocessing/bc5cdr-preprocessed.json'

with open(bc5cdr_corpus_path, "r") as j:
  bc5cdr_corpus = json.loads(j.read())

# Load gold standard terms 
bc5cdr_keywords_path = '../../1-preprocessing/bc5cdr-keywords.tsv'

with open(bc5cdr_keywords_path, "r") as c:
  bc5cdr_lexes_and_sems = pd.read_csv(c, sep='\t')

bc5cdr_lexes = bc5cdr_lexes_and_sems.lex.to_numpy()
bc5cdr_lexes_and_sems['sem'] = bc5cdr_lexes_and_sems['sem'].str.lower()
bc5cdr_semantic_class_names = bc5cdr_lexes_and_sems['sem'].unique().tolist()

# Print to console
display(bc5cdr_lexes_and_sems)

,lex,sem
0,naloxone_lex,chemical
1,clonidine_lex,chemical
2,hypertensive_lex,disease
3,nalozone_lex,chemical
4,hypotensive_lex,disease
...,...,...
4866,galactose_lex,chemical
4867,dgalactose_lex,chemical
4868,dglucose_lex,chemical
4869,memory_deterioration_lex,disease


## Prepare the GENIA Corpus Data for Analysis

Prepare the corpus vocabulary.

In [3]:
# Compile the BC5CDR corpus vocabulary
pre_vocab = []
for i in range(len(bc5cdr_corpus)):
  pre_vocab.append(bc5cdr_corpus[i].split())

vocab = []
for i in range(len(pre_vocab)):
  for j in range(len(pre_vocab[i])):
    vocab.append(pre_vocab[i][j])

vocab = list(set(vocab))
vocab.sort()

# Helper function to ensure that CountVectorizer doesn't ignore any terms
def analyzer_custom(doc):
  return doc.split()

# Convert bc5cdr documents into term-in-document matrix of token counts.
counter = CountVectorizer(lowercase=False, vocabulary=vocab, analyzer=analyzer_custom)
collection = counter.transform(bc5cdr_corpus)

## Evaluate Term Dispersion Scores for Selected Measures

Calculate bag-of-words model word statistics and related quantities.

In [4]:
# Calculate word statistics and related quantities
m = len(counter.get_feature_names_out()) # vocab size
d = collection.shape[0] # collection size
N_i = wordstats.get_Ni(collection)
N_j = wordstats.get_Nj(collection)
N = wordstats.get_N(N_j)
B_ij = wordstats.get_Bij(collection)
B_i = wordstats.get_Bi(B_ij)
B_j = wordstats.get_Bj(B_ij)
B_i.A[0][11764] = 1459 # Hack to fix idiosyncracy with brentq solver for stopword 'of'
DF = wordstats.get_DF(B_i, d)
CF = wordstats.get_CF(N_i)
nij_by_nj = wordstats.get_nij_by_nj(collection, N_j)
thetas = np.array(range(1, max(N_i.A[0]) + 1))/N
opt_thetas = wordstats.get_opt_thetas(N, m, d, N_i, N_j, B_i, thetas)

Evaluate term dispersion scores.

In [5]:
# Calculate word dispersion scores according the various measures used in this study
IDF = wordstats.get_IDF(DF)
ICF = wordstats.get_ICF(CF)
Chisq = wordstats.get_Chisq(collection)
CG = wordstats.get_CG(N_i, B_i)
ICB = wordstats.get_ICB(nij_by_nj, B_i)
DoP = wordstats.get_DoP(collection, N_i, N_j, N)
RICF = wordstats.get_RICF(opt_thetas, N, ICF)

/Users/pasheridan/Desktop/github-repos/bursty-term-measure/bc5cdr/2-tables/singletons-excluded-analysis/../../../wordstats.py:209: RuntimeWarning: divide by zero encountered in log
  return -np.log(chisq_values)


Arrange term dispersion scores into a data frame.

In [6]:
# Initialize term dispersion scores data frame (augmented with ni and bi values)
term_scores_aug_df = pd.DataFrame(data=
                    {'lex': counter.get_feature_names_out(),
                     'IDF': IDF.A[0],
                     'ICF': ICF.A[0],
                     'Chi-sq': Chisq,
                     'CG': CG.A[0],
                     'ICB': ICB.A[0],
                     'DoP': DoP.A[0],
                     'RICF': RICF.A[0],
                     'bi': B_i.A[0],
                     'ni': N_i.A[0]})

# Augment with low-level and high-level semantic classes
term_scores_aug_df = pd.merge(term_scores_aug_df, bc5cdr_lexes_and_sems, on='lex', how='left')

# Tidy up the data frame
new_order = ['lex', 'sem', 'bi', 'ni', 'IDF', 'ICF', 'Chi-sq', 'CG', 'ICB', 'DoP', 'RICF'] # Define column ordering
term_scores_aug_df = term_scores_aug_df.reindex(columns=new_order)
term_scores_aug_df = term_scores_aug_df.rename(columns={'lex': 'term'}) # Rename 'lex' column to 'term'
all_duplicates = term_scores_aug_df[term_scores_aug_df.duplicated(keep='first')] # There are a few duplicate rows for some unknown reason
print("Duplicate rows:")
display(all_duplicates)
term_scores_aug_df = term_scores_aug_df.drop_duplicates() # Drop any duplicate rows

# Print to console
print("Term dispersion scores:")
display(term_scores_aug_df)

Duplicate rows:


,term,sem,bi,ni,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF


Term dispersion scores:


,term,sem,bi,ni,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
0,0001abstract,NaN,1,1,7.313220,12.557595,0.702910,1.0,278.000000,-0.000978,-0.000371
1,0014unit,NaN,1,3,7.313220,11.458983,680.512989,3.0,480.000000,-0.000563,1.098241
2,001abstract,NaN,3,3,6.214608,11.458983,0.673894,1.0,302.666667,-0.003194,-0.001195
3,0070unit,NaN,1,1,7.313220,12.557595,0.702910,1.0,160.000000,-0.000563,-0.000371
4,0075mgkg,NaN,1,1,7.313220,12.557595,0.702910,1.0,100.000000,-0.000352,-0.000371
...,...,...,...,...,...,...,...,...,...,...,...
17950,zuclopenthixol_lex,chemical,1,1,7.313220,12.557595,0.702910,1.0,131.000000,-0.000461,-0.000371
17951,zung,NaN,1,1,7.313220,12.557595,0.702910,1.0,200.000000,-0.000704,-0.000371
17952,zungconde,NaN,1,2,7.313220,11.864448,234.217595,2.0,260.000000,-0.000457,0.692776
17953,zyban_lex,chemical,1,3,7.313220,11.458983,680.512989,3.0,447.000000,-0.000524,1.098241


In [7]:
# Integrate KeyBERT scores

# Load KeyBERT results 
keybert_keywords_path = '../keybert-scores.tsv'

with open(keybert_keywords_path, "r") as c:
  keybert_scores_df = pd.read_csv(c, sep='\t')

# Replace blanks by NANs in sems column
keybert_scores_df['sem'] = keybert_scores_df['sem'].fillna(str())

# Convert semantic classes to lowercase
keybert_scores_df['sem'] = keybert_scores_df['sem'].str.lower()

# Add KeyBERT scores to main results df
term_scores_aug_df = pd.merge(term_scores_aug_df, keybert_scores_df, on=['term', 'sem'], how='left')
term_scores_aug_df['KeyBERT'] = term_scores_aug_df['KeyBERT'].fillna(0)

# Check for duplicate terms
all_duplicates = keybert_scores_df[keybert_scores_df.duplicated(subset=['term', 'sem'], keep='last')]
print("Duplicate rows:")
display(all_duplicates)
keybert_scores_df = keybert_scores_df.drop_duplicates(subset=['term', 'sem'], keep='last')

# Reorder columns
term_scores_aug_df = term_scores_aug_df.reindex(columns=['term', 'sem', 'bi', 'ni', 'IDF', 'ICF', 'Chi-sq', 'CG', 'ICB', 'DoP', 'KeyBERT', 'RICF'])

# Check for duplicate terms
all_duplicates = term_scores_aug_df[term_scores_aug_df.duplicated(keep='first')]
print("Duplicate rows:")
display(all_duplicates)
term_scores_aug_df = term_scores_aug_df.drop_duplicates() # Drop any duplicate rows

display(term_scores_aug_df)

Duplicate rows:


,term,sem,KeyBERT


Duplicate rows:


,term,sem,bi,ni,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,RICF


,term,sem,bi,ni,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,RICF
0,0001abstract,NaN,1,1,7.313220,12.557595,0.702910,1.0,278.000000,-0.000978,0.000000,-0.000371
1,0014unit,NaN,1,3,7.313220,11.458983,680.512989,3.0,480.000000,-0.000563,0.000000,1.098241
2,001abstract,NaN,3,3,6.214608,11.458983,0.673894,1.0,302.666667,-0.003194,0.000000,-0.001195
3,0070unit,NaN,1,1,7.313220,12.557595,0.702910,1.0,160.000000,-0.000563,0.000000,-0.000371
4,0075mgkg,NaN,1,1,7.313220,12.557595,0.702910,1.0,100.000000,-0.000352,0.000000,-0.000371
...,...,...,...,...,...,...,...,...,...,...,...,...
17950,zuclopenthixol_lex,chemical,1,1,7.313220,12.557595,0.702910,1.0,131.000000,-0.000461,0.000667,-0.000371
17951,zung,NaN,1,1,7.313220,12.557595,0.702910,1.0,200.000000,-0.000704,0.000000,-0.000371
17952,zungconde,NaN,1,2,7.313220,11.864448,234.217595,2.0,260.000000,-0.000457,0.000000,0.692776
17953,zyban_lex,chemical,1,3,7.313220,11.458983,680.512989,3.0,447.000000,-0.000524,0.000667,1.098241


In [8]:
# Integrate KeyLLM scores

# Load KeyLLM results 
keyllm_keywords_path = '../keyllm-scores.tsv'

with open(keyllm_keywords_path, "r") as c:
  keyllm_scores_df = pd.read_csv(c, sep='\t')

# Remove rows with empty term entries
keyllm_scores_df = keyllm_scores_df.dropna(subset=['term'])

# Replace blanks by NANs in sems column
keyllm_scores_df['sem'] = keyllm_scores_df['sem'].fillna(str())

# Convert semantic classes to lowercase
keyllm_scores_df['sem'] = keyllm_scores_df['sem'].str.lower()

# Check for duplicate terms
all_duplicates = keyllm_scores_df[keyllm_scores_df.duplicated(subset=['term', 'sem'], keep='last')]
print("Duplicate rows:")
display(all_duplicates)
keyllm_scores_df = keyllm_scores_df.drop_duplicates(subset=['term', 'sem'], keep='last')

# Print to console
print("KeyLLM scores:")
display(keyllm_scores_df)

# Add KeyLLM scores to main results df
term_scores_aug_df = pd.merge(term_scores_aug_df, keyllm_scores_df, on=['term', 'sem'], how='left')
term_scores_aug_df['KeyLLM'] = term_scores_aug_df['KeyLLM'].fillna(0)

# Reorder columns
term_scores_aug_df = term_scores_aug_df.reindex(columns=['term', 'sem', 'bi', 'ni', 'IDF', 'ICF', 'Chi-sq', 'CG', 'ICB', 'DoP', 'KeyBERT', 'KeyLLM', 'RICF'])

# Check for duplicate terms
all_duplicates = term_scores_aug_df[term_scores_aug_df.duplicated(keep='first')]
print("Duplicate rows:")
display(all_duplicates)
term_scores_aug_df = term_scores_aug_df.drop_duplicates() # Drop any duplicate rows

display(term_scores_aug_df)

Duplicate rows:


,term,sem,KeyLLM


KeyLLM scores:


,term,sem,KeyLLM
1,112dihydro2acenaphthylenylpiperazine_lex,chemical,0.000667
2,11deoxycortisol_lex,chemical,0.000667
3,11dichloro222trifluoroethane_lex,chemical,0.000667
4,11ketopregnenolone_sulphate_lex,chemical,0.000667
5,125dihydroxyvitamin_d_lex,chemical,0.000667
...,...,...,...
9101,methicillin_lex,chemical,0.000000
9102,tazobactam_lex,chemical,0.000000
9103,galactose_lex,chemical,0.000000
9104,dgalactose_lex,chemical,0.000000


Duplicate rows:


,term,sem,bi,ni,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF


,term,sem,bi,ni,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
0,0001abstract,NaN,1,1,7.313220,12.557595,0.702910,1.0,278.000000,-0.000978,0.000000,0.000000,-0.000371
1,0014unit,NaN,1,3,7.313220,11.458983,680.512989,3.0,480.000000,-0.000563,0.000000,0.000000,1.098241
2,001abstract,NaN,3,3,6.214608,11.458983,0.673894,1.0,302.666667,-0.003194,0.000000,0.000000,-0.001195
3,0070unit,NaN,1,1,7.313220,12.557595,0.702910,1.0,160.000000,-0.000563,0.000000,0.000000,-0.000371
4,0075mgkg,NaN,1,1,7.313220,12.557595,0.702910,1.0,100.000000,-0.000352,0.000000,0.000000,-0.000371
...,...,...,...,...,...,...,...,...,...,...,...,...,...
17950,zuclopenthixol_lex,chemical,1,1,7.313220,12.557595,0.702910,1.0,131.000000,-0.000461,0.000667,0.000667,-0.000371
17951,zung,NaN,1,1,7.313220,12.557595,0.702910,1.0,200.000000,-0.000704,0.000000,0.000000,-0.000371
17952,zungconde,NaN,1,2,7.313220,11.864448,234.217595,2.0,260.000000,-0.000457,0.000000,0.000000,0.692776
17953,zyban_lex,chemical,1,3,7.313220,11.458983,680.512989,3.0,447.000000,-0.000524,0.000667,0.000667,1.098241


In [9]:
# Write scores data frame to TSV
term_scores_aug_df.to_csv('term-dispersion-scores.tsv', sep='\t', index=False)

## Compile BC5CDR Corpus Summary Statistics

This is the result of Table 4 from the manuscript.

In [10]:
# Count number of distinct lexical units in each semantic class
lex_counts = term_scores_aug_df.dropna(subset=['sem']).groupby('sem')['term'].nunique().reindex(bc5cdr_semantic_class_names).to_list()

# Count number of annotations associated with each semantic class
annotation_counts = term_scores_aug_df.dropna(subset=['sem']).groupby('sem')['ni'].sum().reindex(bc5cdr_semantic_class_names).to_list()

# Count number of singletons associated with each semantic class
singleton_counts = term_scores_aug_df[term_scores_aug_df['ni'] == 1].dropna(subset=['sem']).groupby('sem')['ni'].sum().reindex(bc5cdr_semantic_class_names).to_list()

# Initialize BC5CDR summary statistics data frame
bc5cdr_summary_stats_df = pd.DataFrame({
    'Semantic class': bc5cdr_semantic_class_names,
    'Unique terms': lex_counts,
    'Annotations': annotation_counts,
    'Singletons': singleton_counts
})

# Print BC5CDR summary statistics to console
display(bc5cdr_summary_stats_df)

,Semantic class,Unique terms,Annotations,Singletons
0,chemical,2086,15819,720
1,disease,2757,12630,1347


## Terminology Extraction Task Experiment

Here we reproduce the result of the BC5CDR analysis from the manuscript.

In [11]:
# Create a minimal data frame of term dispersion scores
term_scores_df = term_scores_aug_df[term_scores_aug_df['ni'] > 1] # Filter out singletons
term_scores_df = term_scores_df.reset_index(drop=True) # Reinitialize row indices
term_scores_df = term_scores_df.drop(columns=['sem', 'ni', 'bi'])

# Print to console
display(term_scores_df)

# Write scores data frame to TSV
term_scores_aug_df.to_csv('term-dispersion-scores-minimal.tsv', sep='\t', index=False)

,term,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
0,0014unit,7.313220,11.458983,680.512989,3.0,480.000000,-0.000563,0.000000,0.000000,1.098241
1,001abstract,6.214608,11.458983,0.673894,1.0,302.666667,-0.003194,0.000000,0.000000,-0.001195
2,04fold,7.313220,11.864448,234.217595,2.0,416.000000,-0.000732,0.000000,0.000000,0.692776
3,1000mm3,6.620073,11.864448,0.688299,1.0,239.000000,-0.001682,0.000000,0.000000,-0.000786
4,100mgkg,6.620073,10.948157,inf,2.5,635.000000,-0.001819,0.000000,0.000000,0.915504
...,...,...,...,...,...,...,...,...,...,...
10688,zones,6.214608,11.458983,0.673894,1.0,201.666667,-0.002128,0.000000,0.000000,-0.001195
10689,zonisamide_lex,6.620073,10.765836,inf,3.0,414.500000,-0.000806,0.001333,0.001333,1.097826
10690,zungconde,7.313220,11.864448,234.217595,2.0,260.000000,-0.000457,0.000000,0.000000,0.692776
10691,zyban_lex,7.313220,11.458983,680.512989,3.0,447.000000,-0.000524,0.000667,0.000667,1.098241


Define various functions used in the analysis.

In [12]:
# Grab the top k terms
def top_k(dct, k):
  keys = dct.keys()
  values = []
  for key in keys:
    values.append(dct[key][:k])
  keys_values_pair = zip(keys, values)
  return dict(keys_values_pair)

# Count up terms
def count_words(lst, imp_words):
  counter = 0
  for x in lst:
    if x in imp_words:
      counter += 1
  return counter

# Randomly resort term dispersion scores data frame
def resort(term_scores_df):
  sorted_terms = []
  bursty_measure_names = term_scores_df.columns.values.tolist()[1:]

  for measure in bursty_measure_names:
      # Copy the data frame and add a random column
      temp_df = term_scores_df.copy()
      temp_df['random'] = np.random.rand(len(temp_df))
        
      # Sort by the measure and the random column
      sorted_df = temp_df[['term', measure, 'random']].sort_values(by=[measure, 'random'], ascending=[False, True])
        
      # Append the sorted terms to the list
      sorted_terms.append(np.array(sorted_df['term']))
        
      # Drop the random column from the temporary data frame
      temp_df.drop(columns='random', inplace=True)
    
  sorted_terms = np.array(sorted_terms)
  measure_term_pair = zip(bursty_measure_names, sorted_terms)
  sorted_measures = dict(measure_term_pair)
    
  return sorted_measures
    
# Calculate Precision at k scores
def calc_pk(lex_units, sorted_measures, k_values):
  measures = sorted_measures.keys()
  counts = [[], [], [], [], [], [], [], [], [], []]
  pk_dct = dict(zip(measures, counts))  
  for measure in pk_dct.keys():
      for k in k_values:
          pk_dct[measure].append(count_words(top_k(sorted_measures, k)[measure], lex_units)/k)
  result = pd.DataFrame(pk_dct, index=k_values)
  return result

# Calculate Recall at k scores
def calc_rk(lex_units, sorted_measures, k_values):
  measures = sorted_measures.keys()
  counts = [[], [], [], [], [], [], [], [], [], []]
  rk_dct = dict(zip(measures, counts))  
  for measure in rk_dct.keys():
      for k in k_values:
          rk_dct[measure].append(count_words(top_k(sorted_measures, k)[measure], lex_units)/len(lex_units))
  result = pd.DataFrame(rk_dct, index=k_values)
  return result

# Calculate F1 at k scores
def calc_fk(pk, rk):
  result = 2 * (pk * rk) / (pk + rk)
  result = result.fillna(0) # Nan scores are redefined as 0
  return result

# Calculate Rank Biased Overlap scores
def calc_rbo(sorted_measures, k_values):
  RICF = sorted_measures["RICF"]
  measures = sorted_measures.keys()
  counts = [[], [], [], [], [], [], [], [], [], []]
  rbo_dct = dict(zip(measures, counts))  
  for measure in rbo_dct.keys():
      for k in k_values:
          S = top_k(sorted_measures, k)[measure]
          T = RICF[0:k]
          rbo_dct[measure].append(rbo.RankingSimilarity(S, T).rbo())
  result = pd.DataFrame(rbo_dct, index=k_values)
  return result

# Calculate Rank Biased Overlap scores for each semantic class
def calc_rbo2(sorted_measures, categories):
  measures = sorted_measures.keys()
  counts = [[], [], [], [], [], [], [], [], [], []]
  rbo_dct = dict(zip(measures, counts))

  for measure in measures:
      l1 = sorted_measures[measure].tolist()
      
      for category, lex_units in categories.items():
          S = sorted(set(l1) & set(lex_units), key = l1.index)
          l2 = sorted_measures["RICF"].tolist()
          RICF = sorted(set(l2) & set(lex_units), key = l2.index)          
          rbo_dct[measure].append(rbo.RankingSimilarity(S, RICF).rbo()) 

  result = pd.DataFrame(rbo_dct, index=categories.keys())
  return result
    
# Calculate mean P@k, R@k, and F1@k scores
def calc_score_means(nested_list):
    result = []
    num_outer = len(nested_list)
    num_inner = len(nested_list[0])

    for i in range(num_inner):
        means = {}
        for column in nested_list[0][i].columns:
            values = [nested_list[outer][i][column].values for outer in range(num_outer)]
            mean_values = np.mean(values, axis=0)
            means[column] = mean_values
        result.append(pd.DataFrame(means, index=nested_list[0][i].index))
    return result

# Calculate standard deviations of P@k, R@k, and F1@k scores
def calc_score_sds(nested_list):
    result = []
    num_outer = len(nested_list)
    num_inner = len(nested_list[0])

    for i in range(num_inner):
        std_devs = {}
        for column in nested_list[0][i].columns:
            values = [nested_list[outer][i][column].values for outer in range(num_outer)]
            std_values = np.std(values, axis=0, ddof=1)
            std_devs[column] = std_values
        result.append(pd.DataFrame(std_devs, index=nested_list[0][i].index))
    return result

# Calculate mean RBO scores
def calc_mean_rbo_scores(scores_list):
  R = len(scores_list) # Number of replicates
  H = len(scores_list[0]) # Number of dispersion metrics
  d_metrics = scores_list[0].columns # Dispersion metrics by name
  k_values = scores_list[0].index # Top k values
  result = {}
    
  for d_metric in d_metrics:
    scores = [scores_list[r][d_metric].values for r in range(R)]
    mean_scores = np.mean(scores, axis=0)
    result[d_metric] = mean_scores
        
  return pd.DataFrame(result, index=k_values)

Evaluate Precision at k, Recall at k, F1 at k, and RBO scores.

In [13]:
# Initialize a random seed to ensure results can be replicated
np.random.seed(509303)

# Set number of replicates
R = 100 # To test, set to 5

# These are the Precision @ k, Recall @ k and Rank Biased Overlap scores
all_pk_scores = []
all_rk_scores = []
all_fk_scores = []
all_rbo_scores = []
all_rbo_scores2 = []

# Used as inputs for calculating the various scores
k_values = np.array([10, 50, 100, 500, 1000, 5000])
all_lexes = bc5cdr_lexes_and_sems.loc[bc5cdr_lexes_and_sems['sem'].isin(['chemical', 'disease']), 'lex'].tolist()
chemical_lexes = bc5cdr_lexes_and_sems.loc[bc5cdr_lexes_and_sems['sem'] == 'chemical', 'lex'].tolist()
disease_lexes = bc5cdr_lexes_and_sems.loc[bc5cdr_lexes_and_sems['sem'] == 'disease', 'lex'].tolist()

categories = {
    'all': all_lexes,
    'chemical': chemical_lexes,
    'disease': disease_lexes}

# Calculate evaluation metrics
for r in tqdm(range(R)):
    print('r =', r)
    pk_scores = []
    rk_scores = []
    fk_scores = []
    sorted_measures = resort(term_scores_df)
    
    for category, lex_units in categories.items():
        pk = calc_pk(lex_units, sorted_measures, k_values)
        pk_scores.append(pk)
        rk = calc_rk(lex_units, sorted_measures, k_values)
        rk_scores.append(rk)
        fk = calc_fk(pk, rk)
        fk_scores.append(fk)
    
    all_pk_scores.append(pk_scores)
    all_rk_scores.append(rk_scores)
    all_fk_scores.append(fk_scores)
    rbo_scores = calc_rbo(sorted_measures, k_values)
    all_rbo_scores.append(rbo_scores)
    rbo_scores2 = calc_rbo2(sorted_measures, categories)
    all_rbo_scores2.append(rbo_scores2)

  0%|                                                                                                                                                | 0/100 [00:00<?, ?it/s]

r = 0


  1%|█▎                                                                                                                                      | 1/100 [00:07<12:47,  7.76s/it]

r = 1


  2%|██▋                                                                                                                                     | 2/100 [00:15<12:51,  7.87s/it]

...

 98%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▎  | 98/100 [12:48<00:15,  7.92s/it]

r = 98


 99%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▋ | 99/100 [12:56<00:07,  7.97s/it]

r = 99


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [13:04<00:00,  7.84s/it]


Save evaluation metrics as Pkl files.

In [14]:
# Ensure the directory exists
os.makedirs('scores-dump', exist_ok=True)

# Write P@k scores to Pkl
with open('scores-dump/all_pk_scores.pkl', 'wb') as file:
    pickle.dump(all_pk_scores, file)

# Write R@k scores to Pkl
with open('scores-dump/all_rk_scores.pkl', 'wb') as file:
    pickle.dump(all_rk_scores, file)

# Write F1@k scores to Pkl
with open('scores-dump/all_fk_scores.pkl', 'wb') as file:
    pickle.dump(all_fk_scores, file)

# Write RBO scores to Pkl
with open('scores-dump/all_rbo_scores.pkl', 'wb') as file:
    pickle.dump(all_rbo_scores, file)

# Write RBO scores as calculated for each semantic class to Pkl
with open('scores-dump/all_rbo_scores2.pkl', 'wb') as file:
    pickle.dump(all_rbo_scores2, file)

In [15]:
# Calculate mean P@k scores and write to CSV
all_pk_scores_means = calc_score_means(all_pk_scores)
os.makedirs('table-13', exist_ok=True)
pd.DataFrame(all_pk_scores_means[0]).to_csv('table-13/all-pk-means.csv', index=False)
pd.DataFrame(all_pk_scores_means[1]).to_csv('table-13/chemical-pk-means.csv', index=False)
pd.DataFrame(all_pk_scores_means[2]).to_csv('table-13/disease-pk-means.csv', index=False)

# Calculate standard deviations for P@k scores and write to CSV
all_pk_scores_sds = calc_score_sds(all_pk_scores)
os.makedirs('table-a5', exist_ok=True)
pd.DataFrame(all_pk_scores_sds[0]).to_csv('table-a5/all-pk-sds.csv', index=False)
pd.DataFrame(all_pk_scores_sds[1]).to_csv('table-a5/chemical-pk-sds.csv', index=False)
pd.DataFrame(all_pk_scores_sds[2]).to_csv('table-a5/disease-pk-sds.csv', index=False)

In [16]:
# Display mean P@k scores and console
print("Mean P@k scores:")
with pd.option_context('display.precision', 4):
    display(all_pk_scores_means[0].round(4))
    display(all_pk_scores_means[1].round(4))
    display(all_pk_scores_means[2].round(4))

Mean P@k scores:


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.4420,0.2720,0.6030,0.6780,0.7000,0.7000,1.000,1.0000,0.6760
50,0.4462,0.2870,0.5780,0.6620,0.5600,0.8400,1.000,1.0000,0.6596
100,0.4493,0.2824,0.5782,0.7433,0.6500,0.7400,1.000,1.0000,0.7439
500,0.4521,0.2794,0.5711,0.6742,0.6060,0.6300,1.000,1.0000,0.6768
1000,0.4517,0.2780,0.5703,0.6282,0.5590,0.5466,1.000,1.0000,0.6229
5000,0.3613,0.2765,0.4205,0.4262,0.3636,0.3755,0.526,0.5075,0.4300


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.2440,0.1150,0.4230,0.5780,0.5000,0.2000,0.3000,0.2000,0.5760
50,0.2518,0.1134,0.4048,0.5520,0.4400,0.3600,0.4306,0.4270,0.5480
100,0.2505,0.1085,0.4070,0.6459,0.5100,0.3300,0.4200,0.4321,0.6439
500,0.2542,0.1085,0.4065,0.5635,0.4900,0.2980,0.4546,0.4522,0.5632
1000,0.2543,0.1078,0.4053,0.4868,0.4260,0.2802,0.4439,0.4482,0.4776
5000,0.1867,0.1116,0.2392,0.2421,0.2166,0.1871,0.2533,0.2468,0.2440


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.1980,0.1570,0.1800,0.1000,0.200,0.5000,0.7000,0.8000,0.1000
50,0.1944,0.1736,0.1732,0.1100,0.120,0.4800,0.5694,0.5730,0.1116
100,0.1988,0.1739,0.1712,0.0974,0.140,0.4100,0.5800,0.5679,0.1000
500,0.1979,0.1709,0.1645,0.1107,0.116,0.3320,0.5454,0.5478,0.1137
1000,0.1975,0.1702,0.1650,0.1414,0.133,0.2665,0.5561,0.5518,0.1453
5000,0.1746,0.1649,0.1813,0.1841,0.147,0.1884,0.2728,0.2606,0.1860


In [17]:
# Displaye standard deviation of P@k scores to console
print("P@k scores standard deviations:")
with pd.option_context('display.precision', 4):
    display(all_pk_scores_sds[0].round(4))
    display(all_pk_scores_sds[1].round(4))
    display(all_pk_scores_sds[2].round(4))

P@k scores standard deviations:


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.1640,0.1326,0.1439,0.0416,0.0,0.0000,0.0000,0.0000,0.0429
50,0.0652,0.0660,0.0688,0.0206,0.0,0.0000,0.0000,0.0000,0.0197
100,0.0467,0.0462,0.0467,0.0047,0.0,0.0000,0.0000,0.0000,0.0049
500,0.0201,0.0166,0.0181,0.0048,0.0,0.0000,0.0000,0.0000,0.0057
1000,0.0114,0.0105,0.0107,0.0042,0.0,0.0005,0.0000,0.0000,0.0020
5000,0.0005,0.0011,0.0001,0.0010,0.0,0.0001,0.0013,0.0018,0.0008


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.1321,0.0978,0.1332,0.0416,0.0,0.0000,0.0000,0.0000,0.0429
50,0.0620,0.0428,0.0741,0.0209,0.0,0.0000,0.0100,0.0096,0.0218
100,0.0391,0.0317,0.0485,0.0059,0.0,0.0000,0.0000,0.0074,0.0049
500,0.0170,0.0113,0.0167,0.0058,0.0,0.0000,0.0035,0.0062,0.0049
1000,0.0101,0.0074,0.0099,0.0033,0.0,0.0006,0.0043,0.0058,0.0017
5000,0.0003,0.0010,0.0000,0.0006,0.0,0.0001,0.0011,0.0013,0.0006


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.1392,0.1037,0.1044,0.0000,0.0,0.0000,0.0000,0.0000,0.0000
50,0.0488,0.0495,0.0585,0.0179,0.0,0.0000,0.0100,0.0096,0.0187
100,0.0379,0.0352,0.0376,0.0044,0.0,0.0000,0.0000,0.0074,0.0000
500,0.0152,0.0130,0.0130,0.0038,0.0,0.0000,0.0035,0.0062,0.0039
1000,0.0097,0.0084,0.0077,0.0031,0.0,0.0005,0.0043,0.0058,0.0016
5000,0.0004,0.0008,0.0001,0.0008,0.0,0.0000,0.0008,0.0011,0.0007


In [18]:
# Calculate mean R@k scores and write to CSV
all_rk_scores_means = calc_score_means(all_rk_scores)
os.makedirs('table-14', exist_ok=True)
pd.DataFrame(all_rk_scores_means[0]).to_csv('table-14/all-rk-means.csv', index=False)
pd.DataFrame(all_rk_scores_means[1]).to_csv('table-14/chemical-rk-means.csv', index=False)
pd.DataFrame(all_rk_scores_means[2]).to_csv('table-14/disease-rk-means.csv', index=False)

# Calculate standard deviations for R@k scores and write to CSV
all_rk_scores_sds = calc_score_sds(all_rk_scores)
os.makedirs('table-a6', exist_ok=True)
pd.DataFrame(all_rk_scores_sds[0]).to_csv('table-a6/all-rk-sds.csv', index=False)
pd.DataFrame(all_rk_scores_sds[1]).to_csv('table-a6/chemical-rk-sds.csv', index=False)
pd.DataFrame(all_rk_scores_sds[2]).to_csv('table-a6/disease-rk-sds.csv', index=False)

In [19]:
# Display mean R@k scores console
print("Mean R@k scores:")
with pd.option_context('display.precision', 4):
    display(all_rk_scores_means[0].round(4))
    display(all_rk_scores_means[1].round(4))
    display(all_rk_scores_means[2].round(4))

Mean R@k scores:


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0009,0.0006,0.0012,0.0014,0.0014,0.0014,0.0021,0.0021,0.0014
50,0.0046,0.0029,0.0059,0.0068,0.0057,0.0086,0.0103,0.0103,0.0068
100,0.0092,0.0058,0.0119,0.0153,0.0133,0.0152,0.0205,0.0205,0.0153
500,0.0464,0.0287,0.0586,0.0692,0.0622,0.0647,0.1026,0.1026,0.0695
1000,0.0927,0.0571,0.1171,0.1290,0.1148,0.1122,0.2053,0.2053,0.1279
5000,0.3709,0.2838,0.4317,0.4374,0.3732,0.3854,0.5400,0.5209,0.4413


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0012,0.0005,0.0020,0.0027,0.0024,0.0009,0.0014,0.0009,0.0027
50,0.0060,0.0027,0.0096,0.0131,0.0104,0.0085,0.0102,0.0101,0.0130
100,0.0119,0.0051,0.0193,0.0306,0.0241,0.0156,0.0199,0.0204,0.0305
500,0.0601,0.0257,0.0962,0.1333,0.1159,0.0705,0.1076,0.1070,0.1333
1000,0.1203,0.0510,0.1918,0.2304,0.2016,0.1326,0.2101,0.2121,0.2260
5000,0.4418,0.2640,0.5660,0.5728,0.5125,0.4427,0.5993,0.5840,0.5773


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0007,0.0006,0.0007,0.0004,0.0007,0.0018,0.0025,0.0029,0.0004
50,0.0035,0.0031,0.0031,0.0020,0.0022,0.0087,0.0103,0.0104,0.0020
100,0.0072,0.0063,0.0062,0.0035,0.0051,0.0149,0.0210,0.0206,0.0036
500,0.0359,0.0310,0.0298,0.0201,0.0210,0.0602,0.0989,0.0993,0.0206
1000,0.0716,0.0617,0.0598,0.0513,0.0482,0.0966,0.2016,0.2001,0.0527
5000,0.3166,0.2990,0.3287,0.3337,0.2665,0.3416,0.4945,0.4725,0.3372


In [20]:
# Display standard deviation of R@k scores to console
print("R@k scores standard deviations:")
with pd.option_context('display.precision', 4):
    display(all_rk_scores_sds[0].round(4))
    display(all_rk_scores_sds[1].round(4))
    display(all_rk_scores_sds[2].round(4))

R@k scores standard deviations:


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0003,0.0003,0.0003,0.0001,0.0,0.0000,0.0000,0.0000,0.0001
50,0.0007,0.0007,0.0007,0.0002,0.0,0.0000,0.0000,0.0000,0.0002
100,0.0010,0.0009,0.0010,0.0001,0.0,0.0000,0.0000,0.0000,0.0001
500,0.0021,0.0017,0.0019,0.0005,0.0,0.0000,0.0000,0.0000,0.0006
1000,0.0023,0.0022,0.0022,0.0009,0.0,0.0001,0.0000,0.0000,0.0004
5000,0.0005,0.0012,0.0001,0.0010,0.0,0.0001,0.0014,0.0018,0.0008


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0006,0.0005,0.0006,0.0002,0.0,0.0000,0.0000,0.0000,0.0002
50,0.0015,0.0010,0.0018,0.0005,0.0,0.0000,0.0002,0.0002,0.0005
100,0.0018,0.0015,0.0023,0.0003,0.0,0.0000,0.0000,0.0004,0.0002
500,0.0040,0.0027,0.0039,0.0014,0.0,0.0000,0.0008,0.0015,0.0012
1000,0.0048,0.0035,0.0047,0.0016,0.0,0.0003,0.0020,0.0027,0.0008
5000,0.0007,0.0023,0.0000,0.0014,0.0,0.0002,0.0025,0.0031,0.0015


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0005,0.0004,0.0004,0.0000,0.0,0.0000,0.0000,0.0000,0.0000
50,0.0009,0.0009,0.0011,0.0003,0.0,0.0000,0.0002,0.0002,0.0003
100,0.0014,0.0013,0.0014,0.0002,0.0,0.0000,0.0000,0.0003,0.0000
500,0.0028,0.0024,0.0024,0.0007,0.0,0.0000,0.0006,0.0011,0.0007
1000,0.0035,0.0030,0.0028,0.0011,0.0,0.0002,0.0016,0.0021,0.0006
5000,0.0006,0.0015,0.0002,0.0014,0.0,0.0000,0.0014,0.0020,0.0013


In [21]:
# Calculate mean F1@k scores and write to CSV
all_fk_scores_means = calc_score_means(all_fk_scores)
os.makedirs('table-15', exist_ok=True)
pd.DataFrame(all_fk_scores_means[0]).to_csv('table-15/all-fk-means.csv', index=False)
pd.DataFrame(all_fk_scores_means[1]).to_csv('table-15/chemical-fk-means.csv', index=False)
pd.DataFrame(all_fk_scores_means[2]).to_csv('table-15/disease-fk-means.csv', index=False)

# Calculate standard deviations for F1@k scores and write to CSV
all_fk_scores_sds = calc_score_sds(all_fk_scores)
os.makedirs('table-a7', exist_ok=True)
pd.DataFrame(all_fk_scores_sds[0]).to_csv('table-a7/all-fk-sds.csv', index=False)
pd.DataFrame(all_fk_scores_sds[1]).to_csv('table-a7/chemical-fk-sds.csv', index=False)
pd.DataFrame(all_fk_scores_sds[2]).to_csv('table-a7/disease-fk-sds.csv', index=False)

In [22]:
# Display mean F1@k scores to console
print("Mean F1@k scores:")
with pd.option_context('display.precision', 4):
    display(all_fk_scores_means[0].round(4))
    display(all_fk_scores_means[1].round(4))
    display(all_fk_scores_means[2].round(4))

Mean F1@k scores:


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0018,0.0011,0.0025,0.0028,0.0029,0.0029,0.0041,0.0041,0.0028
50,0.0091,0.0058,0.0117,0.0135,0.0114,0.0171,0.0203,0.0203,0.0134
100,0.0181,0.0114,0.0233,0.0299,0.0262,0.0298,0.0402,0.0402,0.0299
500,0.0842,0.0520,0.1063,0.1255,0.1128,0.1173,0.1862,0.1862,0.1260
1000,0.1539,0.0947,0.1943,0.2140,0.1904,0.1862,0.3407,0.3407,0.2122
5000,0.3661,0.2801,0.4260,0.4317,0.3684,0.3804,0.5329,0.5141,0.4356


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0023,0.0011,0.0040,0.0054,0.0047,0.0019,0.0028,0.0019,0.0054
50,0.0116,0.0052,0.0187,0.0255,0.0203,0.0166,0.0199,0.0197,0.0253
100,0.0226,0.0098,0.0368,0.0584,0.0461,0.0298,0.0380,0.0391,0.0582
500,0.0973,0.0415,0.1556,0.2157,0.1875,0.1140,0.1740,0.1731,0.2155
1000,0.1634,0.0693,0.2604,0.3127,0.2737,0.1800,0.2852,0.2880,0.3068
5000,0.2625,0.1568,0.3363,0.3403,0.3045,0.2630,0.3560,0.3470,0.3430


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0014,0.0011,0.0013,0.0007,0.0014,0.0036,0.0051,0.0058,0.0007
50,0.0069,0.0062,0.0062,0.0039,0.0043,0.0171,0.0203,0.0204,0.0040
100,0.0139,0.0122,0.0120,0.0068,0.0098,0.0287,0.0406,0.0397,0.0070
500,0.0608,0.0525,0.0505,0.0340,0.0356,0.1019,0.1674,0.1681,0.0349
1000,0.1051,0.0906,0.0878,0.0752,0.0708,0.1418,0.2960,0.2937,0.0773
5000,0.2251,0.2126,0.2337,0.2373,0.1895,0.2428,0.3516,0.3360,0.2398


In [23]:
# Display standard deviation of F1@k scores to console
print("F1@k scores standard deviations:")
with pd.option_context('display.precision', 4):
    display(all_fk_scores_sds[0].round(4))
    display(all_fk_scores_sds[1].round(4))
    display(all_fk_scores_sds[2].round(4))

F1@k scores standard deviations:


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0007,0.0005,0.0006,0.0002,0.0,0.0000,0.0000,0.0000,0.0002
50,0.0013,0.0013,0.0014,0.0004,0.0,0.0000,0.0000,0.0000,0.0004
100,0.0019,0.0019,0.0019,0.0002,0.0,0.0000,0.0000,0.0000,0.0002
500,0.0037,0.0031,0.0034,0.0009,0.0,0.0000,0.0000,0.0000,0.0011
1000,0.0039,0.0036,0.0036,0.0014,0.0,0.0002,0.0000,0.0000,0.0007
5000,0.0005,0.0012,0.0001,0.0010,0.0,0.0001,0.0013,0.0018,0.0008


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0012,0.0009,0.0013,0.0004,0.0,0.0000,0.0000,0.0000,0.0004
50,0.0029,0.0020,0.0034,0.0010,0.0,0.0000,0.0005,0.0004,0.0010
100,0.0035,0.0029,0.0044,0.0005,0.0,0.0000,0.0000,0.0007,0.0004
500,0.0065,0.0043,0.0064,0.0022,0.0,0.0000,0.0013,0.0024,0.0019
1000,0.0065,0.0048,0.0064,0.0021,0.0,0.0004,0.0028,0.0037,0.0011
5000,0.0004,0.0013,0.0000,0.0008,0.0,0.0001,0.0015,0.0018,0.0009


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0010,0.0007,0.0008,0.0000,0.0,0.0000,0.0000,0.0000,0.0000
50,0.0017,0.0018,0.0021,0.0006,0.0,0.0000,0.0004,0.0003,0.0007
100,0.0027,0.0025,0.0026,0.0003,0.0,0.0000,0.0000,0.0005,0.0000
500,0.0047,0.0040,0.0040,0.0012,0.0,0.0000,0.0011,0.0019,0.0012
1000,0.0052,0.0045,0.0041,0.0016,0.0,0.0003,0.0023,0.0031,0.0008
5000,0.0005,0.0011,0.0001,0.0010,0.0,0.0000,0.0010,0.0014,0.0009


In [24]:
# Calculate mean RBO scores
mean_rbo_scores = calc_mean_rbo_scores(all_rbo_scores)

# Write to CSV
os.makedirs('table-16', exist_ok=True)
pd.DataFrame(mean_rbo_scores).to_csv('table-16/rbo-means.csv', index=False)

# Display RBO scores to console
with pd.option_context('display.precision', 4):
    display(mean_rbo_scores.round(4))

,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0013,0.0000,0.0067,0.8641,0.3220,0.0000,0.0000,0.0000,1.0
50,0.0097,0.0000,0.0147,0.9026,0.5571,0.0000,0.0000,0.0000,1.0
100,0.0177,0.0000,0.0282,0.9164,0.6240,0.0000,0.0000,0.0000,1.0
500,0.0764,0.0000,0.1395,0.9378,0.7294,0.0110,0.0159,0.0113,1.0
1000,0.1425,0.0000,0.2792,0.9398,0.7537,0.0652,0.0596,0.0528,1.0
5000,0.4921,0.1796,0.7068,0.9434,0.7958,0.4438,0.3619,0.3536,1.0


In [25]:
# Calculate mean RBO scores by semantic class
mean_rbo_scores2 = calc_mean_rbo_scores(all_rbo_scores2)

# Write to CSV
os.makedirs('table-17', exist_ok=True)
pd.DataFrame(mean_rbo_scores2).to_csv('table-17/rbo-means-by-semantic-class.csv', index=False)

# Display RBO scores to console
with pd.option_context('display.precision', 4):
    display(mean_rbo_scores2.round(4))

,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
all,0.5754,0.3812,0.7644,0.9633,0.8433,0.5339,0.4292,0.4337,1.0
chemical,0.5448,0.3613,0.7029,0.9683,0.8530,0.5011,0.4569,0.4513,1.0
disease,0.6138,0.4201,0.8065,0.9480,0.8063,0.5759,0.3993,0.4043,1.0


## Top 10 Ranked Terms Example

Here we reproduce the result of Table 10 from the manuscript.

In [26]:
# Initialize a random seed to ensure results can be replicated
np.random.seed(411010)

# Retrieve top 10 ranked terms
top = 10
ranked_terms_df = resort(term_scores_df)
top_10_ranked_terms_df = pd.DataFrame(top_k(ranked_terms_df, top))

# Print to console
display(top_10_ranked_terms_df)

# Write to CSV
os.makedirs('table-18', exist_ok=True)
top_10_ranked_terms_df.to_csv('table-18/top-10-terms.csv', index=False)

,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
0,mini,autoantibody,b2,sgk1,artery_calcification_lex,nonalcoholic,toxicity_lex,toxicity_lex,tam_lex
1,hypoparathyroidism_lex,invariable,yohimbine_lex,tam_lex,tam_lex,dissecting_aneurysm_lex,seizures_lex,seizures_lex,sgk1
2,conjugation,postmarketing,t_lex,flecainide_lex,val_lex,nontraumatic,hypertension_lex,hypotension_lex,rizatriptan_lex
3,p20,spreading,renal,rizatriptan_lex,adenovirus_disease_lex,postinfarction,mg_lex,hypertension_lex,mpoanca
4,lumbosacral,nightmares,extract,mpoanca,ntg_lex,gerstmann_syndrome_lex,hypotension_lex,dopamine_lex,flecainide_lex
5,eluting,uninephrectomy,desogestrel_lex,ht,ht,erythroderma_lex,dopamine_lex,bradycardia_lex,artery_calcification_lex
6,tubulointerstitial_nephritis_lex,desorptionionization,oral_contraceptives_lex,dlsotalol_lex,rizatriptan_lex,leukemoid_reaction_lex,seizure_lex,nephrotoxicity_lex,apap_lex
7,nitricoxide_lex,compromise,sirolimus_lex,apap_lex,ocs_lex,magnesium_sulfate_lex,pain_lex,cocaine_lex,dlsotalol_lex
8,hypersecretory,hydroxyproline_lex,asa_lex,artery_calcification_lex,lozenge,thoracic_hematomyelia_lex,creatinine_lex,myocardial_infarction_lex,ht
9,plasticity,sn_lex,gap43,ax_lex,mpoanca,caproate_lex,bradycardia_lex,cardiotoxicity_lex,cs_lex


## Stopwords Exploratory Analysis

Here we reproduce the result of Table 11 from the manuscript.

In [27]:
def getrank(sorted_measures):
    unique_terms = set()
    for terms in sorted_measures.values():
        unique_terms.update(terms)
    unique_terms = sorted(unique_terms)
    
    # Create a data frame to hold the rankings
    ranking_df = pd.DataFrame(index=unique_terms, columns=sorted_measures.keys())
    
    # Fill the data frame with rankings
    for measure, terms in sorted_measures.items():
        for rank, term in enumerate(terms):
            ranking_df.at[term, measure] = rank + 1  # Rank starts from 1
    
    # Replace NaN with a large number to indicate unranked terms
    ranking_df = ranking_df.fillna(len(unique_terms) + 1)
    #csv_file_path = 'ranking_table.csv'
    #ranking_df.to_csv(csv_file_path)
    return ranking_df

# Function to filter stopwords from the ranking data frame
def filter_stopwords(ranking_df):
    stopwords_list = set(stopwords.words('english'))
    
    # Filter the data frame to include only stopwords
    stopwords_rank = ranking_df[ranking_df.index.isin(stopwords_list)]
    
    # Save the stopwords ranking data frame to a CSV file
    #csv_file_path = 'stopwords_ranking_table.csv'
    #stopwords_rank.to_csv(csv_file_path)
    
    return stopwords_rank

In [28]:
# Initialize a random seed to ensure results can be replicated
np.random.seed(629004)

# Generate term dispersion ranks for R different versions of the data
all_quantiles_df = []
for r in tqdm(range(R)):
    sorted_measures = resort(term_scores_df)
    rank = getrank(sorted_measures)
    stopwords_ranks_df = filter_stopwords(rank)
    bursty_measure_names = stopwords_ranks_df.head(0)
    quantiles = []
    for bursty_measure_name in bursty_measure_names:
        quantiles.append(stopwords_ranks_df[bursty_measure_name].quantile([0, 0.25, 0.5, 0.75, 1]))
    quantiles_df = pd.DataFrame(quantiles)
    all_quantiles_df.append(quantiles_df)

# Extract the column and index names from the first quantiles data frame
columns = all_quantiles_df[0].columns
index = all_quantiles_df[0].index

# Initialize empty data frames to store the mean and standard deviation values
mean_df = pd.DataFrame(index=index, columns=columns)
std_df = pd.DataFrame(index=index, columns=columns)

# Compute the mean and standard deviation of corresponding elements across all matrices
for col in columns:
    for idx in index:
        values = [matrix.at[idx, col] for matrix in all_quantiles_df]
        mean_df.at[idx, col] = np.mean(values)
        std_df.at[idx, col] = np.std(values)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:40<00:00,  2.47it/s]


In [29]:
# Print to console
print("Mean values:")
with pd.option_context('display.precision', 4):
    display(mean_df)
print("\nStandard deviations:")
with pd.option_context('display.precision', 4):
    display(std_df)

# Write to CSV
os.makedirs('table-19', exist_ok=True)
mean_df.to_csv('table-19/stopword-rank-means.csv')
std_df.to_csv('table-19/stopword-rank-sds.csv')

Mean values:


,0.00,0.25,0.50,0.75,1.00
IDF,1224.18,9900.1125,10485.945,10648.5,10693.0
ICF,402.85,9656.9725,10448.97,10641.5,10693.0
Chi-sq,274.86,5019.25,6718.0,7378.75,10682.43
CG,104.0,4570.1225,6270.0,7270.49,10497.11
ICB,88.0,4347.25,6129.5,7585.75,10661.0
DoP,659.0,9852.25,10485.5,10648.5,10693.0
KeyBERT,2647.68,4642.0325,6557.48,8618.13,10614.63
KeyLLM,2505.44,4544.055,6581.58,8545.995,10630.34
RICF,105.43,6144.0,7115.5,7449.6,10689.0



Standard deviations:


,0.00,0.25,0.50,0.75,1.00
IDF,763.8022,5.2435,0.4177,0.0,0.0
ICF,336.3298,7.0892,0.3303,0.0,0.0
Chi-sq,250.6012,0.0,0.0,0.0,0.4951
CG,0.0,29.5799,0.0,0.5291,161.9455
ICB,0.0,0.0,0.0,0.0,0.0
DoP,0.0,0.0,0.0,0.0,0.0
KeyBERT,73.6724,306.11,362.4252,349.2302,77.7921
KeyLLM,75.8384,305.8478,383.5129,329.6366,53.6148
RICF,1.1336,0.0,0.0,0.5809,0.0
